# DeBERTa for PII Detection

This notebook documents the baseline DeBERTa token classification experiment for PII detection.

## 1. Setup

Run the next cell if the environment has not been installed yet.

In [ ]:
# Uncomment if dependencies are not installed yet
# !pip install -r requirements.txt
# !pip install -r models/transformer/requirements.txt
# !pip install sentencepiece

import os
from pathlib import Path
import json

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / 'models').exists() else CURRENT_DIR.parent
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Notebook working directory:', Path.cwd().resolve())

In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Train the baseline DeBERTa model

This configuration matches the baseline DeBERTa run stored in the repository metrics.

In [ ]:
!python models/transformer/train_deberta.py \
  --artifact-prefix deberta_overnight_20260608 \
  --output-dir models/transformer/deberta-pii-overnight \
  --epochs 5 \
  --batch-size 2 \
  --gradient-accumulation-steps 8 \
  --learning-rate 1e-5 \
  --weight-decay 0.01 \
  --warmup-ratio 0.1 \
  --max-length 256 \
  --chunk-size 128 \
  --max-grad-norm 0.5 \
  --loss-mode sqrt_balanced \
  --save-total-limit 1

## 3. View the training metrics

In [ ]:
metrics_path = PROJECT_ROOT / 'results/metrics/deberta_overnight_20260608_metrics.json'
with metrics_path.open('r', encoding='utf-8') as f:
    metrics = json.load(f)

print(json.dumps({
    'token_f1': metrics['token_level']['f1'],
    'entity_f1': metrics['entity_level']['f1'],
    'total_tokens': metrics['total_tokens'],
}, indent=2))

## 4. Re-evaluate the saved model

In [ ]:
!python models/transformer/evaluate_saved_transformer.py \
  --model-key deberta \
  --model-dir models/transformer/deberta-pii-overnight \
  --artifact-prefix deberta_recheck

## 5. Run manual inference

In [ ]:
from models.transformer.pipeline import TransformerPIIPipeline

pipeline = TransformerPIIPipeline(
    model_dirs={'deberta': str(PROJECT_ROOT / 'models/transformer/deberta-pii-overnight')}
)

result = pipeline.predict(
    'My name is John Smith, my email is john@example.com, and my website is https://john.dev',
    model_key='deberta',
    min_confidence=0.5,
)

print(result['summary'])
print(result['entities'])
print(result['redacted_text'])